In [ ]:
class SERDataset(torch.utils.data.Dataset):
    def __init__(self, rows, feature_extractor, seconds: float):
        self.rows = rows
        self.fe = feature_extractor
        self.seconds = seconds

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx: int):
        r = self.rows[idx]
        audio = load_audio_16k_mono(r["wav_path"])
        audio = fix_to_seconds(audio, SECONDS)
        x = self.fe(audio, sampling_rate=16000, return_tensors="pt", padding=False)
        return {
            "input_values": x["input_values"].squeeze(0),
            "labels": torch.tensor(int(r["label"]), dtype=torch.long),
        }


@dataclass
class Collator:
    feature_extractor: Any

    def __call__(self, features):
        input_values = [f["input_values"] for f in features]
        labels = torch.stack([f["labels"] for f in features])
        batch = self.feature_extractor.pad(
            {"input_values": input_values}, padding=True, return_tensors="pt"
        )
        batch["labels"] = labels
        return batch


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro", zero_division=0),
    }


def split_stratified(rows, train_ratio=0.7, val_ratio=0.15, test_ratio=0.15, seed=42):
    if len(rows) < 3:
        return rows, [], []
    y = [int(r["label"]) for r in rows]
    try:
        train_rows, tmp_rows = train_test_split(
            rows, test_size=(1.0 - train_ratio), random_state=seed, stratify=y
        )
        vf = val_ratio / (val_ratio + test_ratio) if (val_ratio + test_ratio) > 0 else 0.5
        y_tmp = [int(r["label"]) for r in tmp_rows]
        val_rows, test_rows = train_test_split(
            tmp_rows, test_size=(1.0 - vf), random_state=seed, stratify=y_tmp
        )
    except ValueError:
        rng = random.Random(seed)
        tmp = rows[:]
        rng.shuffle(tmp)
        n = len(tmp)
        n_train = max(1, int(n * train_ratio))
        n_val = max(1, int(n * val_ratio))
        train_rows, val_rows, test_rows = tmp[:n_train], tmp[n_train : n_train + n_val], tmp[n_train + n_val :]
        if not test_rows:
            test_rows = val_rows
    return train_rows, val_rows, test_rows


train_rows, val_rows, test_rows = split_stratified(rows, seed=SEED)
print("train", len(train_rows), "val", len(val_rows), "test", len(test_rows))